# 01 — Exploratory Data Analysis

Explore the RAVDESS and TESS datasets: class distributions, audio lengths, waveform and spectrogram visualisations.

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# ── Paths (adjust as needed) ──────────────────────────────────────────────
RAVDESS_DIR = Path('../data/ravdess')
TESS_DIR    = Path('../data/toronto')

EMOTION_MAP = {
    1: 'neutral', 2: 'calm', 3: 'happy', 4: 'sad',
    5: 'angry', 6: 'fearful', 7: 'disgust', 8: 'surprised',
}

In [ ]:
# ── Build file manifest ───────────────────────────────────────────────────
records = []

for wav in sorted(RAVDESS_DIR.rglob('*.wav')):
    parts = wav.stem.split('-')
    if len(parts) == 7:
        emotion_code = int(parts[2])
        records.append({'path': wav, 'dataset': 'RAVDESS', 'emotion': EMOTION_MAP[emotion_code]})

for wav in sorted(TESS_DIR.rglob('*.wav')):
    tail = wav.stem.split('_')[-1].lower()
    tess_map = {'neutral':'neutral','happy':'happy','sad':'sad','angry':'angry',
                'fear':'fearful','disgust':'disgust','ps':'surprised'}
    if tail in tess_map:
        records.append({'path': wav, 'dataset': 'TESS', 'emotion': tess_map[tail]})

df = pd.DataFrame(records)
print(f'Total files: {len(df)}')
df.groupby(['dataset', 'emotion']).size().unstack(fill_value=0)

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (dataset, sub) in zip(axes, df.groupby('dataset')):
    sub['emotion'].value_counts().plot(kind='bar', ax=ax, color=sns.color_palette('tab10'))
    ax.set_title(f'{dataset} — emotion distribution')
    ax.set_xlabel('Emotion')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── Audio duration distribution ───────────────────────────────────────────
sample_files = df.sample(min(200, len(df)), random_state=42)['path'].tolist()
durations = [librosa.get_duration(path=str(p)) for p in sample_files]

plt.figure(figsize=(8, 3))
plt.hist(durations, bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Duration (s)')
plt.ylabel('Count')
plt.title('Audio Duration Distribution (sample n=200)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Waveform and mel-spectrogram visualisation ─────────────────────────────
sample_rows = df.groupby('emotion').first().reset_index()

num_emotions = len(sample_rows)
fig, axes = plt.subplots(num_emotions, 2, figsize=(14, num_emotions * 2.5))

for i, row in sample_rows.iterrows():
    y, sr = librosa.load(str(row['path']), sr=22050, mono=True)
    librosa.display.waveshow(y, sr=sr, ax=axes[i, 0], color='steelblue')
    axes[i, 0].set_title(f"{row['emotion']} — waveform")
    axes[i, 0].set_xlabel('')

    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=40), ref=np.max)
    img = librosa.display.specshow(mel, sr=sr, x_axis='time', y_axis='mel', ax=axes[i, 1], cmap='magma')
    axes[i, 1].set_title(f"{row['emotion']} — mel spectrogram")
    fig.colorbar(img, ax=axes[i, 1], format='%+2.0f dB')

plt.tight_layout()
plt.show()